In [ ]:
!pip install deepeval

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
def fetch_html_body_content(html_url):

    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("⚠ HTML Fail:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # # 移除 Abstract
    # text = re.sub(
    #     r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
    #     '',
    #     text,
    #     flags=re.S|re.I
    # )

    # 截断 Acknowledgment
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]
    return text, "html_success"

In [ ]:
# --- DATA QUALITY FIX: strip arXiv HTML navigation boilerplate ---
# arXiv HTML pages start with a big block of UI elements and a table-of-contents
# (section number + title lines) before the actual article prose. We need to skip
# past all of that to give Llama clean article text.

# Exact strings that appear in the arXiv HTML nav/UI preamble
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

# Substrings — if ANY of these appear anywhere in the line, skip it
BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",            # catches "Back to <any section>"
    "Content selection saved",
    "Describe the issue below",
]

def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble (nav UI + table of contents), then return
    the first max_chars of actual article prose.

    Strategy: scan forward until we find a line that looks like the start of
    real article content (a sentence-length line with lowercase words — i.e.
    actual prose, not a short section heading). Everything before that is
    preamble/TOC and gets dropped.
    """
    lines = full_text.splitlines()

    # Phase 1: Find where the real article prose begins.
    # TOC lines are short (section titles). Real prose lines are longer and
    # contain lowercase words forming sentences.
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        # Skip blanks
        if not stripped:
            continue
        # A "prose" line: reasonably long, contains spaces (multiple words),
        # and has some lowercase characters (not all-caps headings).
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: From body_start onward, drop any remaining boilerplate lines
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            # Preserve paragraph breaks in the body, but skip leading blanks
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        # Skip lines matching known boilerplate substrings
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        # Skip exact boilerplate matches
        if stripped in BOILERPLATE_EXACT:
            continue
        # Skip bare section numbers like "2", "2.1", "A.3"
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import os
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from google.colab import userdata

# 1. API Setup
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Define Metric
judge_metric = GEval(
    name="G-Eval-Judge",
    criteria=(
        f"""
Evaluate the quality of summaries written for a news article.
Rate each summary on four dimensions:
style transfer intensity, content preservation, naturalness and readability.
You should rate on a scale from 1 (worst) to 5 (best).

Provide your evaluation as follows:
Explanation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Explanation:' and 'Total rating:' in your answer.
"""
    ),
    model="gpt-5.2",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT,
    ],
)

# 3. Read Data
df = pd.read_csv("results_llama_math.csv")
scored_data = []

# 4. Processing Loop
# Adjust .head(3) to the number of rows you want to process
for idx, row in df.iloc[44:46].iterrows():
    print(f"--- Processing Row {idx} ---")

    html_url = row.get("html_url")
    article, status = fetch_html_body_content(html_url)

    if status != "html_success":
        print(f"❌ Row {idx} failed: {status}")
        continue

    summary = row["llama_prompt"]
    article_snippet = get_article_snippet(article)

    # Create the test case
    test_case = LLMTestCase(
        input="Summary Evaluation Task",
        actual_output=summary,
        retrieval_context=[article_snippet]
    )

    try:
        # Use .measure() instead of evaluate() to keep it silent
        judge_metric.measure(test_case)

        # Collect results
        row_result = row.to_dict()
        row_result['G-Eval rate'] = judge_metric.score
        row_result['G-Eval Explanation'] = judge_metric.reason
        scored_data.append(row_result)

        # Only print exactly what you want
        print(f"✅ Rate: {judge_metric.score}")
        print(f"✅ Explanation: {judge_metric.reason}\n")

    except Exception as e:
        print(f"❌ Row {idx} evaluation error: {e}")

# 5. Save Results
if scored_data:
    df_output = pd.DataFrame(scored_data)
    output_filename = "results_llama_math_with_scores.csv"
    df_output.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"Finished! File saved as: {output_filename}")
else:
    print("No results were generated.")